# Product Line Profitability & Margin Performance Analysis
## Nassau Candy Distributor — Machine Learning Project

**Objective:** analyze product profitability and build a machine-learning model to classify transactions as **High Margin** or **Low Margin**.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix


## 1. Load and inspect the dataset


In [ ]:
df = pd.read_csv("Nassau Candy Distributor.csv")
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.isnull().sum())


## 2. Calculate the profitability KPI
Margin % = Gross Profit / Sales × 100


In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True, errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True, errors="coerce")
df["Margin %"] = np.where(df["Sales"] != 0, df["Gross Profit"] / df["Sales"] * 100, 0)
print("Total Sales:", round(df["Sales"].sum(),2))
print("Total Gross Profit:", round(df["Gross Profit"].sum(),2))
print("Overall Margin %:", round(df["Gross Profit"].sum()/df["Sales"].sum()*100,2))


## 3. Product, division and regional analysis


In [ ]:
product = df.groupby("Product Name").agg(Orders=("Order ID","count"), Sales=("Sales","sum"), Units=("Units","sum"), Gross_Profit=("Gross Profit","sum")).reset_index()
product["Margin %"] = product["Gross_Profit"]/product["Sales"]*100
display(product.sort_values("Margin %", ascending=False))
division = df.groupby("Division").agg(Sales=("Sales","sum"), Gross_Profit=("Gross Profit","sum"), Units=("Units","sum")).reset_index()
division["Margin %"] = division["Gross_Profit"]/division["Sales"]*100
display(division)
region = df.groupby("Region").agg(Sales=("Sales","sum"), Gross_Profit=("Gross Profit","sum"), Units=("Units","sum")).reset_index()
region["Margin %"] = region["Gross_Profit"]/region["Sales"]*100
display(region)


## 4. Visualization


In [ ]:
product.sort_values("Margin %").plot(x="Product Name", y="Margin %", kind="barh", legend=False, figsize=(10,6))
plt.xlabel("Margin (%)")
plt.title("Product Line Margin Performance")
plt.tight_layout()
plt.show()


## 5. Feature engineering and time-based split
The model does **not** use Sales, Cost, Gross Profit or Margin as input features because those values directly determine the target and would create data leakage.


In [ ]:
d = df.sort_values("Order Date").copy()
d["year"] = d["Order Date"].dt.year
d["month"] = d["Order Date"].dt.month
d["quarter"] = d["Order Date"].dt.quarter
d["dayofweek"] = d["Order Date"].dt.dayofweek
split_date = d["Order Date"].quantile(0.80)
train = d[d["Order Date"] <= split_date].copy()
test = d[d["Order Date"] > split_date].copy()
threshold = train["Margin %"].median()
train["High_Margin"] = (train["Margin %"] >= threshold).astype(int)
test["High_Margin"] = (test["Margin %"] >= threshold).astype(int)
print("Training rows:", len(train), "Testing rows:", len(test))
print("High-margin threshold:", threshold)


## 6. Train Random Forest classifier


In [ ]:
features = ["Ship Mode","Country/Region","Division","Region","Units","year","month","quarter","dayofweek","Product Name"]
X_train, y_train = train[features], train["High_Margin"]
X_test, y_test = test[features], test["High_Margin"]
categorical = [c for c in features if X_train[c].dtype == "object"]
numeric = [c for c in features if c not in categorical]
preprocessor = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), categorical), ("num","passthrough",numeric)])
model = Pipeline([("preprocessor",preprocessor),("model",RandomForestClassifier(n_estimators=300,max_depth=12,min_samples_leaf=3,class_weight="balanced",random_state=42,n_jobs=-1))])
model.fit(X_train, y_train)


## 7. Evaluate the model


In [ ]:
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:,1]
print("Accuracy:", accuracy_score(y_test,pred))
print("ROC-AUC:", roc_auc_score(y_test,prob))
print(classification_report(y_test,pred))
print(confusion_matrix(y_test,pred))


## 8. Predict profitability class for a new transaction


In [ ]:
new_transaction = pd.DataFrame([{"Ship Mode":"Standard Class","Country/Region":"United States","Division":"Chocolate","Region":"Pacific","Units":4,"year":2025,"month":6,"quarter":2,"dayofweek":2,"Product Name":"Wonka Bar - Nutty Crunch Surprise"}])
prediction = model.predict(new_transaction)[0]
probability = model.predict_proba(new_transaction)[0,1]
print("Prediction:", "High Margin" if prediction==1 else "Low Margin")
print("Probability of High Margin:", round(probability*100,2), "%")
